# 公式LLM-X dataset: raw scan / token inspection

現在のrawファイルをread-onlyで検査し、raw → preprocessing → prompt → local tokenizerを確認します。
このNotebook単独で上から実行できます。もう一方のデータ調査は [02_external_data_and_token_inspection.ipynb](02_external_data_and_token_inspection.ipynb) に分離しています。
data/の変更・ダウンロード・モデル推論・API呼び出しは行いません。結果はメモリ上に保持します。

全体scan → 一覧 → 保存inventoryとの任意比較 → 選択record → 対象範囲のtoken計測・比較の順です。


## 1. imports / path

scan対象directory、実ファイルのschemaを取得する関数、reader、preprocessing、token counterを明示します。


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dataset_adapters").is_dir() and (p / "configs/sources.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("workspace直下またはnotebooks/から起動してください。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools.inventory_llmx_data import create_inventory
from dataset_adapters.base import inventory_context
from dataset_adapters.registry import get_adapter
from tools.live_inspection import (
    official_frames, compare_frames, show_frame,
    token_summary_frame, compare_saved_tokens,
)
from tools.notebook_inspection import (
    select_record, preview_raw, preprocessing_for_inspection,
    selected_query, token_record, read_batch_summary,
)
from tools.count_input_tokens import resolve_encoding
from tools.token_inventory import build_token_inventory

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)
print("Official raw:", ROOT / "data/llmx_data")
print("No API clients. Scan/token results stay in memory; saved inventories are not overwritten.")


## 2. LIVE SCAN: official raw

create_inventory()がoffline_data以下の全episode NPZをallow_pickle=Falseで読みます。
checkpoint等の非episodeファイルはmetadataのみ。shapeは実配列どおりで、singleton次元を削除しません。


In [ ]:
llmx_inventory = create_inventory(ROOT / "data/llmx_data")
print("LIVE:", llmx_inventory["task_count"], "tasks,",
      llmx_inventory["episode_count"], "episodes,",
      llmx_inventory["total_timesteps"], "timesteps")


## 3. official task overview

以下の3 DataFrameは、上で作ったllmx_inventory objectのみから整形します。

### official_tasks_df：カラムの意味

1行は1タスクの集計です。全公式タスクを対象とし、この表で統計を再計算せずcreate_inventory()の結果を取り出します。

| カラム | 意味・計算方法 |
| --- | --- |
| `task` | タスク名。例：`MountainCar-v0`。 |
| `number_of_episodes` | そのタスクのエピソードNPZファイル数。`len(files)`。1ファイルを1エピソードとして数える。 |
| `total_timesteps` | 各NPZの`states.shape[0]`を合計したステップ数。秒数ではない。長さが取得できないファイルは合計から除外する。 |
| `timesteps_min` | エピソード長（`states.shape[0]`）の最小値。 |
| `timesteps_max` | エピソード長の最大値。 |
| `timesteps_mean` | エピソード長の算術平均。長さが取得できたファイルが対象。 |
| `state_dimensions` | 各`states`配列の第0軸以外の長さの積を取り、重複を除いて並べたリスト。`[200, 4]`なら4。 |
| `action_dimensions` | 各`actions`配列の第0軸以外の長さの積を取り、重複を除いて並べたリスト。 |
| `action_kind` | ファイルごとの判定結果の種類一覧。actionを1ステップ1要素に並べられ、全値が有限かつ整数相当なら`discrete`、それ以外は`continuous`。環境の正式なaction space定義ではなく、値からの判定。 |
| `npz_keys` | そのタスクのNPZに存在するキー名の和集合。全ファイルに全キーがあるという意味ではない。 |
| `schema_consistent_except_T` | 時系列配列の第0軸を`T`に置き換えたschemaが全ファイルで同じか。キー・dtype・残りのshapeを比較し、`episodic_return`のshapeは置き換えない。 |


In [ ]:
official_tasks_df, official_files_df, official_arrays_df = official_frames(llmx_inventory)
show_frame(official_tasks_df, title="LIVE official_tasks_df — 1 task / row")


## 4. official全episode / array shape

official_files_dfは1 episode/file=1行、official_arrays_dfは1 file×1 NPZ key=1行です。
**全行を省略せずスクロール表示**します。任意の絞り込みもDataFrame.query()で可能です。

### official_files_df：カラムの意味

1行は1つのエピソードNPZファイルです。全公式タスクのファイルをまとめています。

| カラム | 意味・計算方法 |
| --- | --- |
| `task` | ファイルが属するタスク名。 |
| `episode_id` | タスク内のファイルを`relative_path`の文字列順に並べて0から振った番号。タスクごとに0へ戻る。ファイル名から取り出した数字ではない。 |
| `filename` | NPZのファイル名。例：`episode_0.npz`。 |
| `relative_path` | inventory作成時のデータルート（既定：`data/llmx_data`）からNPZまでの相対パス。 |
| `timesteps` | そのNPZの`states.shape[0]`。stateがない場合や0次元の場合は未取得（None/NaN）。 |
| `size_bytes` | ディスク上のNPZファイル自体のサイズ（バイト）。展開後の配列の容量ではない。 |
| `schema_group` | 正規化したschemaのSHA256先頭16桁。同じIDなら同じschemaグループ。時系列長を`T`として比較するため、長さが違っても同じIDになり得る。raw配列は変更しない。 |

### official_arrays_df：カラムの意味

1行は1つのNPZ内の1配列です。例えばNPZに4つの配列キーがあれば、そのファイルについて4行できます。配列の実値ではなく検査結果を表示します。

| カラム | 意味・計算方法 |
| --- | --- |
| `task` | 配列を含むNPZが属するタスク名。 |
| `episode_id` | `official_files_df`と同じ、タスク内の相対パス順の0-based番号。単独では全タスクを通した一意IDではない。 |
| `filename` | 配列を含むNPZのファイル名。 |
| `relative_path` | データルートからNPZまでの相対パス。`key`と組み合わせて配列を特定する。 |
| `key` | NPZ内の配列名。例：`states`、`actions`、`rewards`、`episodic_return`。 |
| `shape` | 実配列の各軸の長さ。例：`[200, 4]`。長さ1の軸も省略しない。スカラーは`[]`。 |
| `dtype` | 配列の要素のデータ型。例：`float64`、`int64`。 |
| `ndim` | 配列の軸数。`[200, 4]`なら2。stateの特徴量数とは異なる。スカラーは0。 |
| `num_elements` | 配列の全要素数（`array.size`）。`[200, 4]`なら800。バイト数ではない。スカラーは1。 |


In [ ]:
show_frame(official_files_df, title="LIVE official_files_df — all episodes")
show_frame(official_arrays_df, title="LIVE official_arrays_df — all file × NPZ key rows")
# 例: official_arrays_df.query("task == 'MountainCar-v0' and key == 'states'")


## 5. LIVE SCAN vs SAVED INVENTORY — secondary validation

ここで初めて保存済みinventoryを読みます。ファイルがない場合もlive inspectionは可能です。
全件のkeyでouter joinし、新規／削除／shape・dtype変更・件数差を隠しません。
一致しなくてもrawを変更したりcacheを上書きしたりしません。


In [ ]:
saved_base = ROOT / "outputs/dataset_inventory"
raw_validation = {}
if (saved_base / "llmx_schema.json").is_file():
    saved_llmx = json.loads((saved_base / "llmx_schema.json").read_text())
    _, saved_official_files, saved_official_arrays = official_frames(saved_llmx)
    raw_validation["official_files"] = compare_frames(
        official_files_df, saved_official_files, ["task", "relative_path"], ["timesteps", "size_bytes", "schema_group"])
    raw_validation["official_arrays"] = compare_frames(
        official_arrays_df, saved_official_arrays, ["task", "relative_path", "key"], ["shape", "dtype", "ndim", "num_elements"])
    display(pd.DataFrame([{"comparison": name, "rows": len(table),
                          "matches": int(table.matches.sum()), "differences": int((~table.matches).sum())}
                         for name, table in raw_validation.items()]))
    for name, table in raw_validation.items():
        if not table.matches.all():
            show_frame(table.loc[~table.matches], title=name + " differences", page_size=30)
else:
    print("Saved inventory not found. LIVE scan results remain the primary source.")


## 6. drill-down selection

全体を見た後、ここでdatasetを選びます。番号は0-basedです。
選択を変更したら「selected raw record」から「selected query token」まで再実行してください。


In [ ]:
DATASET = "MountainCar-v0"
EPISODE = 0
SEQUENCE_ID = None
ROW_ID = None
INDEX = 10

HISTORY_SIZE = 1
METRIC = None
QUESTION_NAME = None  # None/Noneでそのtaskの既存LLM-X既定質問。
MODEL = "gpt-4o"
PREPROCESSING_CONFIG = None


## 7. selected raw record — RAW DATA

readerには同じlive inventoryを渡します。保存済みcacheをprimary sourceへ戻しません。
source file、選択sequence、全raw fieldのshape/dtypeと実値を表示します。


In [ ]:
if DATASET not in {t["task"] for t in llmx_inventory["tasks"]}:
    raise ValueError("このNotebookの対象外datasetです。もう一方のNotebookを使用してください。")
with inventory_context(llmx_inventory, {}):
    adapter = get_adapter(DATASET)
sequence, raw_record = select_record(adapter, episode=EPISODE, sequence_id=SEQUENCE_ID, row_id=ROW_ID, index=INDEX)
if sequence is None:
    print("NOT_ACQUIRED: raw inspection unavailable.")
else:
    print("RAW RECORD")
    print("Source:", sequence.source_file, "\nSequence:", sequence.sequence_id, sequence.key)
    print("Episode:", raw_record["episode_id"], "Row:", raw_record.get("row_id"), "Index:", raw_record["index"])
    print("SHA256:", raw_record["source_file_sha256"])
    raw_fields_df = pd.DataFrame([{"field": name, **spec} for name, spec in raw_record["fields"].items()])
    show_frame(raw_fields_df, title="Selected raw fields")
    print(json.dumps(raw_record["fields"], ensure_ascii=False, indent=2))
    raw_preview = preview_raw(adapter, sequence)
    if isinstance(raw_preview, dict):
        for key, value in raw_preview.items():
            print("\n", key, "(episode scope)" if key == "episodic_return" else "[:5]")
            print(value)
    else:
        display(raw_preview)


## 8. PREPROCESSING → HISTORY

RAWとMODEL INPUT AFTER PREPROCESSINGは別です。
公式は既存LLM-X関数のwrapperを使い、元promptの意味を変更しません。


evaluation.py のプロンプト構築部分に対応するのは、その後に呼ばれる build_prompt_queries() です。


Notebook
  preprocessing_for_inspection() → 設定を読む
  selected_query()
    → queries_for_sequence()
      → build_prompt_queries()  → 公式関数でプロンプトを作る

| 処理 | 公式 `evaluate_episode()` | 本project `build_prompt_queries()` |
|---|---|---|
| タスク・質問の解決 | `resolve_task()` / `resolve_question()` | 同じ関数 |
| system promptの構築 | `system_prompt()` | 同じ関数 |
| 対象index・履歴範囲の決定 | `_query_indices()` / `_history_range()` | 同じ関数 |
| 質問文の構築 | `render_question()` | 同じ関数 |
| user promptの構築 | `user_prompt()` | 同じ関数 |
| モデルへの問い合わせ | `backend.complete()` | **しない** |
| 応答の採点 | `_score_response()` | **しない** |
| 戻り値 | 評価結果 `EvaluationResult` | プロンプト情報 `PromptQuery` を順次生成 |

In [ ]:
config, preprocessing_status = preprocessing_for_inspection(DATASET, PREPROCESSING_CONFIG)
print(preprocessing_status)
query = None
if config is not None and sequence is not None:
    display(config)
    try:
        query = selected_query(adapter, sequence, raw_record["index"], config, HISTORY_SIZE, METRIC, QUESTION_NAME)
    except (ValueError, IndexError) as exc:
        print("Model input: N/A —", exc)
    if query is not None:
        print("MODEL INPUT AFTER PREPROCESSING:", config["prompt_mode"])
        print("Metric / question:", query.metric, query.question_name)
        print("H:", query.history_size, "history [start,end):", query.history_start, query.history_end)
        print("\nHISTORY\n", query.history_text)
else:
    print("Token count: N/A")


## 9. actual prompt全文

同じpreprocessingが作ったSYSTEM PROMPTとUSER PROMPT。文字列を表示するだけで送信しません。


In [ ]:
if query is not None:
    print("Prompt mode:", config["prompt_mode"], "Preprocessing:", config["name"])
    print("\nSYSTEM PROMPT\n")
    print(query.system_prompt)
    print("\nUSER PROMPT\n")
    print(query.user_prompt)
else:
    print("Prompt: N/A")


## 10. selected query token

CLI共通count_queryを使用。contentは実文字列のlocal tokenize結果、API入力はcontent+9の既存概算でserver usageではありません。
history/questionはBPE境界のためuser_tokensに単純加算できません。fallback使用時は明示します。


In [ ]:
selected_tokens = None
if query is not None:
    encoding, fallback = resolve_encoding(MODEL)
    selected_tokens = token_record(query, sequence, config, encoding, fallback, MODEL)
    print("Model:", MODEL, "Tokenizer:", encoding.name, "Fallback:", fallback)
    names = ["system_tokens", "history_tokens", "question_tokens", "user_tokens",
             "total_content_tokens", "estimated_api_input_tokens"]
    display(pd.DataFrame([{"component": k, "tokens": selected_tokens[k]} for k in names]))
else:
    print("Token count: N/A")


| component | 意味 |
|---|---|
| `system_tokens` | タスク説明など、system promptのtoken数 |
| `history_tokens` | 履歴部分だけの参考値。すでにuserに含まれる |
| `question_tokens` | 質問部分だけの参考値。すでにuserに含まれる |
| `user_tokens` | 導入文・履歴・質問を含むuser prompt全文 |
| `total_content_tokens` | `system_tokens + user_tokens` |
| `estimated_api_input_tokens` | 本文合計に、メッセージ形式の概算9 tokensを追加 |

## 11. LIVE TOKEN COUNT: official tasks

build_token_inventory()をCLIと共有し、今回の実行でprompt文字列を作りlocal tokenizeします。選択recordとは独立です。
公式のみを対象に、対応taskの全valid queryを計測します。元prompt未対応taskは理由付きskipします。
結果はmemoryに保持し、保存済みbatchを上書きしません。


In [ ]:
TOKEN_MODEL = "gpt-4o"
TOKEN_HISTORY_SIZES = [1]


In [ ]:
with inventory_context(llmx_inventory, {}):
    result = build_token_inventory(
        datasets="all", history_sizes=TOKEN_HISTORY_SIZES, model=TOKEN_MODEL,
    )
print("LIVE count:", result.metadata["number_of_datasets"], "datasets,",
      result.metadata["number_of_queries"], "query/history/static rows")
print("Tokenizer:", result.metadata["tokenizer"], "Fallback:", result.metadata["fallback_encoding_used"])
token_scopes_df = pd.DataFrame(result.scopes)
token_skipped_df = pd.DataFrame(result.skipped)
show_frame(token_scopes_df, title="LIVE scopes — full selected scope vs deterministic sample")
show_frame(token_skipped_df, title="Skipped datasets and reasons")


## 12. all_token_queries_df

1 query / history window / static record=1行。全計測行をmemory上のDataFrameとして保持します。
画面はページ表示だけですが、all_token_queries_df.query(...)で全件を直接検索できます。
episode_id / sequence_id / row_idは非該当ならNAです。


In [ ]:
all_token_queries_df = pd.DataFrame(result.queries)
for col in ["episode_id", "sequence_id", "row_id", "query_index"]:
    all_token_queries_df[col] = all_token_queries_df[col].astype("Int64")
QUERY_START = 0
query_columns = [
    "dataset_id", "source_file", "episode_id", "sequence_id", "row_id", "query_index",
    "history_size", "prompt_mode", "preprocessing_name", "preprocessing_config_sha256",
    "estimated_api_input_tokens", "system_tokens", "user_tokens", "history_tokens", "question_tokens",
    "count_scope", "eligible_query_pool", "sample_size", "seed",
]
show_frame(all_token_queries_df[query_columns], title="LIVE all_token_queries_df", start=QUERY_START, page_size=100)
# 例: all_token_queries_df.query("dataset_id == 'MountainCar-v0' and episode_id == 0")
token_by_sequence_df = pd.DataFrame(result.by_sequence)


## 13. token_by_dataset_df

dataset / mode / H / 前処理設定ごとに分離します。異なるprompt semanticsやsample scopeを混ぜて合算しません。
stdは母標準偏差、p95はnearest-rank（既存CLIと共通）です。


In [ ]:
token_by_dataset_df = token_summary_frame(result)
summary_columns = [
    "dataset_id", "prompt_mode", "preprocessing_name", "history_size", "count_scope",
    "eligible_query_pool", "sample_size", "seed", "counted_queries", "mean", "std",
    "min", "median", "p95", "max", "total",
]
show_frame(token_by_dataset_df[summary_columns], title="LIVE token_by_dataset_df")


## 14. dataset間token distribution

prompt_modeごとに図を分け、各datasetのpreprocessing_name・H・全件/sampleをラベルに表示します。
boxplotのひげは1.5×IQR、点はその外側。幅や点の数から未計測poolの分布を推定しません。


In [ ]:
import matplotlib.pyplot as plt
for mode, mode_df in all_token_queries_df.groupby("prompt_mode", sort=True):
    grouped = list(mode_df.groupby(["dataset_id", "preprocessing_name", "history_size", "count_scope"], sort=True))
    labels = [f"{key[0]}\n{key[1]} | H={key[2]} | {key[3]}" for key, _ in grouped]
    values = [group["estimated_api_input_tokens"].to_numpy() for _, group in grouped]
    fig, ax = plt.subplots(figsize=(13, max(3, 0.65 * len(grouped))))
    ax.boxplot(values, vert=False)
    ax.set_yticks(range(1, len(labels) + 1), labels)
    ax.set_xlabel("Estimated API input tokens (content + 9)")
    ax.set_title(f"Prompt mode: {mode}")
    fig.tight_layout()
    plt.show()


## 15. largest/smallest token queries → raw/prompt drill-down

以下の行を見て「drill-down selection」の選択値を変更してください。
前処理設定・H・metric/questionも一致させ、raw表示から選択queryのtoken計測まで再実行すると同じpromptを確認できます。
各query行に実効設定JSONとSHA256を保持しています。


In [ ]:
largest_token_queries_df = all_token_queries_df.nlargest(20, "estimated_api_input_tokens")
smallest_token_queries_df = all_token_queries_df.nsmallest(20, "estimated_api_input_tokens")
drill_columns = query_columns + ["metric", "question_name"]
show_frame(largest_token_queries_df[drill_columns], title="Largest 20 counted queries")
show_frame(smallest_token_queries_df[drill_columns], title="Smallest 20 counted queries")
example = largest_token_queries_df.iloc[0]
print("Example selection — copy into drill-down selection:")
print("DATASET =", repr(example["dataset_id"]))
print("EPISODE =", int(example["episode_id"]) if pd.notna(example["episode_id"]) else None)
print("SEQUENCE_ID =", int(example["sequence_id"]))
print("ROW_ID =", int(example["row_id"]) if pd.notna(example["row_id"]) else None)
print("INDEX =", int(example["query_index"]))
print("HISTORY_SIZE =", int(example["history_size"]))
print("Metric/question:", example["metric"], example["question_name"])
print("Effective preprocessing JSON:", example["preprocessing_config"])


## 16. optional: Saved batch resultsとのvalidation

read_batch_summary()はここだけで使用します。LIVE結果がprimaryです。
同じdataset・mode・H・設定SHA・model/tokenizer・sampling条件の集計値を比較し、savedのみのHも明示します。
比較は集計値レベルです。prompt全文の厳密一致を調べる場合はquery-levelのprompt SHAを照合してください。
savedがなければ表示するだけで、追加batchは起動しません。


In [ ]:
saved_token_summary = read_batch_summary()
saved_metadata_path = ROOT / "outputs/token_counts/summary.json"
if saved_token_summary is None or not saved_metadata_path.is_file():
    print("Token batch has not been executed yet.")
else:
    saved_token_summary = saved_token_summary.loc[saved_token_summary["dataset_id"].isin({t["task"] for t in llmx_inventory["tasks"]})].copy()
    saved_metadata = json.loads(saved_metadata_path.read_text())
    saved_token_validation_df = compare_saved_tokens(result, saved_token_summary, saved_metadata)
    show_frame(saved_token_validation_df, title="LIVE vs SAVED token aggregates")
    print("Matched aggregates:", int(saved_token_validation_df.matches.sum()))
    print("Saved results read only; no CSV/JSON was overwritten.")
